# Block 2 — Raw-schema checks

This notebook **verifies** the schema produced by `pipeline/build_schema.py`
(`pipeline/schema.json`) — the contract that says *exactly which columns a Tier-1
raw Qualtrics export must contain and on what scale each item is answered*.

It does **not** define the schema (that's `build_schema.py`); it independently
confirms the schema is correct against the project's sources of truth:

1. **Columns match the example export** — same non-system columns as
   `raw_data_deposit/example_raw_export.csv` (should be **52**).
2. **Scales are well-formed** — 42 sliders (0–100), 1 integer donation (0–10),
   1 binary newsletter.
3. **All 13 scored outcomes are traceable** — every raw component each outcome
   needs is present in the export.
4. **Cross-check against R** — codebook ↔ `clean_lib.R` rename map agree;
   every `tier1_required` column is derivable; demographic labels match
   `submission_spec.R`.
5. **Integration test (the real proof)** — build a synthetic raw file using
   *only* the schema, run the actual `scripts/clean.R` on it, and confirm it
   produces a valid cleaned Tier-1 file with all required columns and 17 conditions.

A green run means: *if Block 4 emits these columns, `make clean` will accept it.*

In [ ]:
import csv, json, subprocess, sys, tempfile, random, shutil
from pathlib import Path

# Repo root = folder that contains codebook.csv (works whether the kernel's cwd
# is the repo root or the notebooks/ subfolder).
ROOT = Path.cwd()
if not (ROOT / "codebook.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / "codebook.csv").exists(), f"can't find repo root from cwd={Path.cwd()}"
PIPE = ROOT / "pipeline"
sys.path.insert(0, str(PIPE))
print("repo root:", ROOT)

# Tiny check harness: records failures, prints PASS/FAIL per check.
failures = []
def check(name, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  -- {detail}" if (detail and not ok) else ""))
    if not ok:
        failures.append((name, detail))

## Step 0 — (Re)build and load the schema\n\nWe regenerate `schema.json` from the codebook, then reload it **from disk** — testing the exact artifact Block 4 will read.

In [ ]:
import build_schema
build_schema.main()                                   # regenerate pipeline/schema.json
schema   = json.loads((PIPE / "schema.json").read_text(encoding="utf-8"))
raw_cols = schema["raw_export_columns"]
elicited = schema["elicited_items"]
demos    = schema["demographics"]
print(f"{len(raw_cols)} raw export columns; {len(elicited)} elicited items; {len(demos)} demographics")

## Check 1 — Columns match the example export\n\nThe set of raw columns the schema expects must equal the non-system columns of the shipped example export (and there should be 52). We also confirm the column *order* matches, so our file looks like a real export.

In [ ]:
hdr = next(csv.reader((ROOT / "raw_data_deposit" / "example_raw_export.csv").open(encoding="utf-8")))
sys_cols = set(schema["system_columns_optional"])
example_non_system = [c for c in hdr if c not in sys_cols]

check("raw_export_columns set == example export (non-system)",
      set(raw_cols) == set(example_non_system),
      f"only_in_schema={set(raw_cols)-set(example_non_system)} | only_in_example={set(example_non_system)-set(raw_cols)}")
check("raw_export_columns count == 52", len(raw_cols) == 52, f"got {len(raw_cols)}")
check("column ORDER matches example export", raw_cols == example_non_system, "order differs (set still ok)")

## Check 1b — Condition labels\n\nThe `condition` column may only hold the 17 canonical titles (control + 16 interventions). These are validated against the live R value in Check 4; here we confirm the schema lists them correctly.

In [ ]:
conds = schema["conditions"]
check("17 conditions listed", len(conds["all"]) == 17, f"got {len(conds['all'])}")
check("control present + 16 interventions",
      conds["control"] == "control" and len(conds["interventions"]) == 16
      and conds["all"] == [conds["control"]] + conds["interventions"])

## Check 2 — Response scales are well-formed\n\nEvery item the model is asked must carry a sane scale: sliders 0–100, the donation 0–10, the newsletter binary Yes/No.

In [ ]:
bad = []
for it in elicited:
    s, k = it["scale"], it["scale"]["kind"]
    if k == "slider"  and not (s.get("min") == 0 and s.get("max") == 100): bad.append((it["qualtrics_label"], "slider != 0-100"))
    elif k == "integer" and not (s.get("min") == 0 and s.get("max") == 10): bad.append((it["qualtrics_label"], "integer != 0-10"))
    elif k == "binary"  and set(s.get("options", [])) != {"Yes", "No"}:      bad.append((it["qualtrics_label"], "binary opts"))
    elif k not in {"slider", "integer", "binary"}:                          bad.append((it["qualtrics_label"], f"unexpected kind {k}"))

check("every elicited item has a valid scale", not bad, str(bad))
n_slider = sum(1 for it in elicited if it["scale"]["kind"] == "slider")
check("scale tally = 42 slider / 1 integer / 1 binary",
      n_slider == 42 and len(elicited) == 44, f"sliders={n_slider}, total={len(elicited)}")

## Check 3 — All 13 scored outcomes are traceable\n\n`make clean` builds the 13 scored outcomes from raw items. Each outcome's raw components must be present in our export, or the outcome can't be computed.

In [ ]:
present = set(raw_cols)
missing_by_outcome = {out: [c for c in spec["components"] if c not in present]
                      for out, spec in schema["constructed"]["scored_outcomes"].items()}
missing_by_outcome = {k: v for k, v in missing_by_outcome.items() if v}
check("all scored outcomes' raw components present in export", not missing_by_outcome, str(missing_by_outcome))
check("exactly 13 scored outcomes defined", len(schema["constructed"]["scored_outcomes"]) == 13)

## Check 4 — Cross-check against the R sources of truth\n\nWe pull the canonical constants straight out of `submission_spec.R` and `clean_lib.R` and assert our codebook-derived schema agrees with them. (Skips gracefully if R isn't on this machine — e.g. on a compute cluster; run this notebook locally where R is installed.)

In [ ]:
rscript = shutil.which("Rscript")
R = None
if not rscript:
    print("[skip] Rscript not found -- run Check 4 & 5 locally (R needed). Static checks above still hold.")
else:
    r_code = r"""
      suppressMessages({ source('scripts/lib/submission_spec.R'); source('scripts/lib/clean_lib.R') })
      library(jsonlite)
      cat(toJSON(list(
        conditions     = sst$conditions,
        outcomes       = sst$outcomes,
        tier1_required = sst$tier1_required,
        moderators     = sst$moderators,
        rename_map     = as.list(.rename_map)
      ), auto_unbox = TRUE))
    """
    res = subprocess.run([rscript, "-e", r_code], cwd=ROOT, capture_output=True, text=True)
    assert res.returncode == 0, res.stderr
    R = json.loads(res.stdout)
    print("pulled from R:", list(R))

In [ ]:
if R:
    # (a) codebook qualtrics->target agrees with clean_lib's .rename_map
    cb = {r["qualtrics_label"]: r["target_label"]
          for r in csv.DictReader((ROOT / "codebook.csv").open(encoding="utf-8"))
          if r["section"].startswith("A.")}
    rmap = {v: k for k, v in R["rename_map"].items()}            # .rename_map is target->qualtrics; invert
    disagree = {q: (cb.get(q), rmap[q]) for q in rmap if cb.get(q) != rmap[q]}
    check("codebook rename == clean_lib .rename_map", not disagree, str(disagree))

    # (b) every tier1_required column is derivable from our raw export
    derivable = set(R["tier1_required"])
    raw_targets = {it["target_label"] for it in elicited} | {d["target_label"] for d in demos}
    raw_targets |= {"profile_id", "condition", "age_band"}                    # assigned + derived
    raw_targets |= set(schema["constructed"]["scored_outcomes"])             # composites
    missing_req = sorted(derivable - raw_targets)
    check("all tier1_required columns derivable from our raw export", not missing_req, f"missing={missing_req}")

    # (c) condition labels match submission_spec sst$conditions exactly
    check("schema conditions == submission_spec sst$conditions",
          schema["conditions"]["all"] == R["conditions"],
          f"schema={schema['conditions']['all']} | R={R['conditions']}")

    # (d) demographic labels match submission_spec moderators
    level_mismatch = {}
    for d in demos:
        if d["scale"]["kind"] != "categorical":
            continue
        ours, theirs = set(d["scale"]["codes"].values()), set(R["moderators"][d["qualtrics_label"]])
        if ours != theirs:
            level_mismatch[d["qualtrics_label"]] = {"ours": sorted(ours), "theirs": sorted(theirs)}
    check("demographic labels match submission_spec moderators", not level_mismatch, str(level_mismatch))

## Check 5 — Integration test: run the real `clean.R`\n\nThe decisive check. Build a synthetic raw file (one row per condition) using **only** `schema.raw_export_columns` with random in-range values, run the actual `scripts/clean.R`, and confirm it produces a clean Tier-1 file with every required column and all 17 conditions. If this passes, the schema is sufficient for `make clean`.

In [ ]:
if R:
    random.seed(0)
    conditions = R["conditions"]
    DEMO_MAPS = build_schema.DEMOGRAPHIC_MAPS

    def cell_value(col, i, cond):
        if col == "profile_id": return f"t{i:05d}"
        if col == "condition":  return cond
        if col in DEMO_MAPS:    return random.choice(list(DEMO_MAPS[col]))   # numeric code
        if col == "year_birth": return str(random.randint(1926, 2008))
        if col == "donation":   return str(random.randint(0, 10))
        if col == "newsletter": return random.choice(["Yes", "No"])
        return str(random.randint(0, 100))                                   # sliders

    rows = [{c: cell_value(c, i, cond) for c in raw_cols}
            for i, cond in enumerate(conditions, 1)]

    with tempfile.TemporaryDirectory() as td:
        ip, op = Path(td) / "raw.csv", Path(td) / "clean.csv"
        with ip.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=raw_cols); w.writeheader(); w.writerows(rows)

        res = subprocess.run([rscript, "scripts/clean.R", str(ip), str(op)],
                             cwd=ROOT, capture_output=True, text=True)
        check("clean.R runs on a schema-built raw file (exit 0)", res.returncode == 0,
              (res.stderr or "").strip()[-800:])

        if res.returncode == 0:
            with op.open(encoding="utf-8") as f:
                cr = csv.DictReader(f); out_cols = cr.fieldnames; out_rows = list(cr)
            req = set(R["tier1_required"])
            check("cleaned file has all tier1_required columns", req.issubset(out_cols),
                  f"missing={sorted(req - set(out_cols))}")
            check("cleaned file has one row per condition (17)", len(out_rows) == len(conditions),
                  f"got {len(out_rows)}")
            check("all 17 conditions survive cleaning",
                  {r["condition"] for r in out_rows} == set(conditions))
            vals = [float(r["trust_multidimensional"]) for r in out_rows if r["trust_multidimensional"] != ""]
            check("primary outcome trust_multidimensional within 0-100", all(0 <= v <= 100 for v in vals))

## Summary

In [ ]:
print("=" * 60)
if failures:
    print(f"BLOCK 2 SCHEMA: {len(failures)} CHECK(S) FAILED")
    for n, d in failures:
        print("  -", n, "::", d)
else:
    print("BLOCK 2 SCHEMA: ALL CHECKS PASSED  \u2705")
assert not failures, f"{len(failures)} check(s) failed"